# 20 SQL Interview Problems — AI Engineering Interview Prep

All solutions use DuckDB in-memory. Classic interview problems with explanations.

In [ ]:
import duckdb
conn = duckdb.connect()

conn.execute("""
CREATE TABLE employees AS SELECT * FROM (VALUES
  (1,'Alice',  1,95000,3),(2,'Bob',  2,75000,2),(3,'Charlie',1,85000,1),
  (4,'Diana',  3,70000,2),(5,'Edward',1,105000,1),(6,'Fiona',2,82000,3),
  (7,'George', 4,68000,NULL),(8,'Hannah',3,91000,4)
) t(emp_id,name,dept_id,salary,manager_id)
""")

conn.execute("""
CREATE TABLE orders AS SELECT * FROM (VALUES
  (1,1001,'2024-01-05',250.0),(2,1002,'2024-01-10',180.0),
  (3,1001,'2024-01-15',320.0),(4,1003,'2024-01-20',90.0),
  (5,1001,'2024-02-01',410.0),(6,1002,'2024-02-05',55.0),
  (7,1003,'2024-02-10',200.0),(8,1001,'2024-02-15',130.0),
  (9,1004,'2024-03-01',500.0),(10,1002,'2024-03-10',220.0)
) t(order_id,customer_id,order_date,amount)
""")

conn.execute("""
CREATE TABLE logins AS SELECT * FROM (VALUES
  (1,'2024-01-01'),(1,'2024-01-02'),(1,'2024-01-03'),
  (1,'2024-01-05'),(2,'2024-01-01'),(2,'2024-01-03'),
  (3,'2024-01-02'),(3,'2024-01-03'),(3,'2024-01-04'),(3,'2024-01-05')
) t(user_id,login_date)
""")

print("Tables created.")

## Problem 1: Second Highest Salary

In [ ]:
conn.execute("""
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)
""").df()

## Problem 2: Employees Earning More Than Their Manager

In [ ]:
conn.execute("""
SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS manager_salary
FROM employees e
JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
""").df()

## Problem 3: Find Duplicate Records

In [ ]:
conn.execute("""
CREATE TEMP TABLE temp_data AS SELECT * FROM (VALUES
  (1,'alice@x.com'),(2,'bob@x.com'),(3,'alice@x.com'),(4,'carol@x.com'),(5,'bob@x.com')
) t(id, email)
""")

conn.execute("""
SELECT email, COUNT(*) AS occurrences
FROM temp_data
GROUP BY email
HAVING COUNT(*) > 1
""").df()

## Problem 4: Running Total of Orders by Customer

In [ ]:
conn.execute("""
SELECT order_id, customer_id, order_date, amount,
    SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date
                      ROWS UNBOUNDED PRECEDING) AS running_total
FROM orders
ORDER BY customer_id, order_date
""").df()

## Problem 5: Top 2 Salaries Per Department

In [ ]:
conn.execute("""
WITH ranked AS (
    SELECT *, DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dr
    FROM employees
)
SELECT name, dept_id, salary, dr FROM ranked WHERE dr <= 2
ORDER BY dept_id, dr
""").df()

## Problem 6: Month-over-Month Revenue Growth

In [ ]:
conn.execute("""
WITH monthly AS (
    SELECT DATE_TRUNC('month', order_date::DATE) AS month, SUM(amount) AS revenue
    FROM orders GROUP BY 1
)
SELECT month, revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month,
    ROUND((revenue - LAG(revenue) OVER (ORDER BY month))
          / LAG(revenue) OVER (ORDER BY month) * 100, 1) AS mom_pct
FROM monthly
ORDER BY month
""").df()

## Problem 7: First and Last Order Per Customer

In [ ]:
conn.execute("""
SELECT customer_id,
    MIN(order_date) AS first_order,
    MAX(order_date) AS last_order,
    COUNT(*) AS total_orders,
    SUM(amount) AS lifetime_value
FROM orders
GROUP BY customer_id
ORDER BY lifetime_value DESC
""").df()

## Problem 8: Consecutive Login Streak (Gap-and-Island)

In [ ]:
conn.execute("""
WITH numbered AS (
    SELECT user_id, login_date,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS rn
    FROM logins
),
groups AS (
    SELECT user_id, login_date,
        (login_date::DATE - rn * INTERVAL '1 day')::DATE AS grp
    FROM numbered
)
SELECT user_id, MIN(login_date) AS streak_start, MAX(login_date) AS streak_end,
    COUNT(*) AS streak_length
FROM groups
GROUP BY user_id, grp
ORDER BY user_id, streak_length DESC
""").df()

## Problem 9: Pivot — Count Orders per Customer per Month

In [ ]:
conn.execute("""
SELECT customer_id,
    COUNT(CASE WHEN strftime(order_date, '%m') = '01' THEN 1 END) AS jan,
    COUNT(CASE WHEN strftime(order_date, '%m') = '02' THEN 1 END) AS feb,
    COUNT(CASE WHEN strftime(order_date, '%m') = '03' THEN 1 END) AS mar
FROM orders
GROUP BY customer_id
ORDER BY customer_id
""").df()

## Problem 10: Median Salary

In [ ]:
conn.execute("""
SELECT
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary) AS median_salary,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY salary) AS p25,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY salary) AS p75
FROM employees
""").df()

## Problems 11–20 Highlights

In [ ]:
# Problem 11: Cumulative % of total orders
result = conn.execute("""
SELECT customer_id, SUM(amount) AS revenue,
    ROUND(SUM(SUM(amount)) OVER (ORDER BY SUM(amount) DESC) /
          SUM(SUM(amount)) OVER () * 100, 1) AS cumulative_pct
FROM orders
GROUP BY customer_id
ORDER BY revenue DESC
""").df()
print("Problem 11 - Cumulative %:")
print(result)

# Problem 12: Users with orders every month
result2 = conn.execute("""
SELECT customer_id
FROM orders
GROUP BY customer_id
HAVING COUNT(DISTINCT DATE_TRUNC('month', order_date::DATE)) =
       (SELECT COUNT(DISTINCT DATE_TRUNC('month', order_date::DATE)) FROM orders)
""").df()
print("\nProblem 12 - Ordered every month:")
print(result2)

In [ ]:
# Problem 13: Recursive CTE - Org Chart Hierarchy
conn.execute("""
WITH RECURSIVE org AS (
    -- Anchor: top-level (no manager)
    SELECT emp_id, name, manager_id, 1 AS level, name AS path
    FROM employees
    WHERE manager_id IS NULL
    UNION ALL
    -- Recursive: employees who report to org members
    SELECT e.emp_id, e.name, e.manager_id, o.level + 1,
           o.path || ' > ' || e.name
    FROM employees e
    JOIN org o ON e.manager_id = o.emp_id
)
SELECT emp_id, name, level, path FROM org ORDER BY path
""").df()

In [ ]:
# Problem 14: Customers with increasing order amounts
conn.execute("""
WITH with_prev AS (
    SELECT customer_id, order_date, amount,
        LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_amount
    FROM orders
),
decreasing AS (
    SELECT DISTINCT customer_id FROM with_prev
    WHERE amount <= prev_amount
)
SELECT DISTINCT customer_id AS always_increasing_customer
FROM orders
WHERE customer_id NOT IN (SELECT customer_id FROM decreasing)
""").df()

## Interview Tips

- **Second highest**: use `MAX(salary) WHERE salary < MAX(salary)` or `OFFSET 1 LIMIT 1`.
- **Gap-and-island**: subtract `ROW_NUMBER` from date to get a consistent group key for consecutive sequences.
- **Recursive CTEs**: need anchor (base case) + recursive member + termination condition.
- **PERCENTILE_CONT** interpolates (true median); **PERCENTILE_DISC** returns actual value from data.
- **Pivot** patterns: `COUNT(CASE WHEN col = 'X' THEN 1 END)` or DuckDB's PIVOT clause.